### Custom Review Inference Pipeline
In this step, we load the trained balanced LSTM model and the saved preprocessing tokenization parameters to build a real-time prediction pipeline for custom patient reviews.

### Fixing the Issue

In [ ]:
import os
import re
import json
import numpy as np
import html  
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.layers import TextVectorization

# 1. Set paths to assets
MODEL_PATH = os.path.join('..', 'models', 'bidi_lstm_balanced_model.keras')
VOCAB_PATH = os.path.join('..', 'models', 'vectorizer_vocabulary.json')

print("Loading optimized Bidirectional LSTM model...")
model = load_model(MODEL_PATH)

print("Loading saved vocabulary list...")
with open(VOCAB_PATH, 'r', encoding='utf-8') as f:
    saved_vocabulary = json.load(f)

# 2. Recreate the layer configuration exactly matching training
VOCAB_SIZE = 10000
MAX_LEN = 150

print("Rebuilding inference TextVectorization layer...")
inference_vectorizer = TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN,
    vocabulary=saved_vocabulary # <-- WE INJECT THE EXACT MEMORY DICTIONARY HERE!
)

def clean_text(text):
    text = text.lower()
    text = html.unescape(text)
    return text.strip()

def predict_custom_review(review_text):
    cleaned = clean_text(review_text)
    
    # Pass the string through our reconstructed layer to get the exact matrix
    # We wrap it in [cleaned] because the layer expects a batch or array input
    vectorized_input = inference_vectorizer([cleaned]).numpy()
    
    # Predict class probabilities (0-9)
    raw_probabilities = model.predict(vectorized_input, verbose=0)
    predicted_class_shifted = np.argmax(raw_probabilities, axis=1)[0]
    
    # Shift back to human scale (1-10)
    final_rating = predicted_class_shifted + 1
    
    print(f"\nReview: \"{review_text}\"")
    print(f"Predicted Rating: {final_rating} / 10 Stars")
    print("-" * 50)
    return final_rating

print("\n Modern Production Inference Pipeline Fully Loaded!")

2026-06-06 17:26:12.801986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading optimized Bidirectional LSTM model...
Loading saved vocabulary list...
Rebuilding inference TextVectorization layer...

 Modern Production Inference Pipeline Fully Loaded!


In [ ]:
# Test 4: Pure, unmitigated positive sentiment with high-frequency praise words
review_perfect = "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
predict_custom_review(review_perfect)

# Test 5: A mild/moderate sentiment to see if it lands in the middle range
review_moderate = "It was okay. It did the job, but nothing special. Average results."
predict_custom_review(review_moderate)


Review: "Excellent medication! Highly recommended. It is the best drug on the market, 100% amazing and wonderful experience."
Predicted Rating: 10 / 10 Stars
--------------------------------------------------

Review: "It was okay. It did the job, but nothing special. Average results."
Predicted Rating: 4 / 10 Stars
--------------------------------------------------


4

In [ ]:
# Test 1: Highly Positive Sentiment
review_positive = "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
predict_custom_review(review_positive)

# Test 2: Highly Negative Sentiment
review_negative = "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
predict_custom_review(review_negative)

# Test 3: Mixed / Conditional Sentiment
review_mixed = "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
predict_custom_review(review_mixed)


Review: "This medication completely changed my life. Within two days, my chronic symptoms vanished completely. No side effects at all!"
Predicted Rating: 10 / 10 Stars
--------------------------------------------------

Review: "Absolutely horrible experience. I suffered terrible nausea and severe headaches within an hour of taking this. Avoid at all costs."
Predicted Rating: 1 / 10 Stars
--------------------------------------------------

Review: "The drug works well enough to manage the pain, but the fatigue makes it almost impossible to stay awake during work hours."
Predicted Rating: 5 / 10 Stars
--------------------------------------------------


5